In [ ]:
# Train the model on 16x16 images for edge computing devices (e.g., STM32)

import time
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_openml

# ==========================================
# 1. Initial Configurations
# ==========================================
# Number of representative templates to generate for each digit (0-9)
# 12 is a good balance between accuracy and memory limits on MCUs.
NUM_TEMPLATES_PER_DIGIT = 12 

# Threshold for binarizing grayscale images (0-255) into pure black and white (0-1)
THRESHOLD = 110

# ==========================================
# 2. Hardware-Optimized Processing Functions (16x16)
# ==========================================

def preprocess_images(images_flat):
    """
    Resize images from the default MNIST resolution (28x28) down to 16x16.
    This drastically reduces the memory footprint from 784 bytes to 256 bytes per image,
    making it ideal for memory-constrained microcontrollers.
    """
    processed = []
    for img in images_flat:
        # Convert flat array to 2D matrix (uint8) for OpenCV
        img_28 = img.reshape(28, 28).astype(np.uint8)
        # Resize to 16x16
        img_16 = cv2.resize(img_28, (16, 16))
        # Flatten back to a 1D array of 256 pixels
        processed.append(img_16.flatten())
    return np.array(processed)


def pack_to_4x64bit(binary_array):
    """
    Pack a 256-bit array (16x16 binary image) into four 64-bit integers.
    This bit-packing technique allows the microcontroller to compare images
    using ultra-fast bitwise operations instead of slow array iterations.
    """
    parts = []
    for i in range(4): # 4 chunks of 64 bits = 256 bits
        chunk = binary_array[i*64: (i+1)*64]
        val = 0
        for bit in chunk:
            val = (val << 1) | int(bit)
        parts.append(val)
    return parts


def get_hamming_distance_16x16(img_parts, template_parts):
    """
    Calculate the Hamming Distance between the input image and a template.
    It uses the XOR bitwise operator to find differences rapidly.
    """
    total_distance = 0
    for i in range(4):
        diff = img_parts[i] ^ template_parts[i]
        total_distance += bin(diff).count('1')
    return total_distance

# ==========================================
# 3. Data Downloading and Preparation
# ==========================================
print("Downloading MNIST dataset (this may take a minute or two)...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')

# Extract features and labels, then convert to NumPy arrays
X_full = mnist.data.values
y_full = mnist.target.astype(int).values

print("Splitting data into 50,000 Train and 20,000 Test sets...")
X_train_raw = X_full[:50000]
y_train = y_full[:50000]

X_test_raw = X_full[50000:]
y_test = y_full[50000:]

print("Resizing images to 16x16 and Binarizing...")
# Shrink images to fit microcontroller limits
X_train_16 = preprocess_images(X_train_raw)
X_test_16 = preprocess_images(X_test_raw)

# Convert grayscale pixels to strictly 0 or 1 based on the threshold
X_train_binary = (X_train_16 > THRESHOLD).astype(int)
X_test_binary = (X_test_16 > THRESHOLD).astype(int)

# ==========================================
# 4. Training Phase (Template Extraction via K-Means)
# ==========================================
print(f"Training (Extracting {NUM_TEMPLATES_PER_DIGIT} templates per digit using K-Means)...")
digit_templates = {}

for digit in range(10):
    # Filter the binary images corresponding to the current digit
    digit_images_binary = X_train_binary[y_train == digit]
    
    # Use K-Means clustering to find common writing styles (clusters) for this digit
    kmeans = KMeans(n_clusters=NUM_TEMPLATES_PER_DIGIT, random_state=42, n_init="auto")
    kmeans.fit(digit_images_binary)
    
    templates = []
    # Convert the cluster centers (averages) back to hard binary templates
    for center in kmeans.cluster_centers_:
        binary_array = (center > 0.5).astype(int)
        # Pack the binary template into 64-bit integers for hardware deployment
        templates.append(pack_to_4x64bit(binary_array))
        
    # Store the packed templates for the current digit
    digit_templates[digit] = templates

# ==========================================
# 5. Final Testing Phase (Simulating MCU Behavior)
# ==========================================
print("\n--- Testing on unseen images ---")
start_time = time.time()
correct_predictions = 0
total_tests = len(X_test_binary)

# Pre-pack all test images to speed up the evaluation loop
packed_test_images = [pack_to_4x64bit(img) for img in X_test_binary]

for i in range(total_tests):
    test_img_parts = packed_test_images[i]
    min_distance = 257  # Maximum possible difference in a 16x16 image is 256
    best_match = -1

    # Compare the input image against all extracted templates
    for digit, templates in digit_templates.items():
        for template in templates:
            distance = get_hamming_distance_16x16(test_img_parts, template)
            if distance < min_distance:
                min_distance = distance
                best_match = digit

    # Check if the prediction matches the ground truth label
    if best_match == y_test[i]:
        correct_predictions += 1
        
    # Display progress
    if (i + 1) % 5000 == 0:
        print(f"Processed {i + 1} / {total_tests} images...")

end_time = time.time()
accuracy = (correct_predictions / total_tests) * 100

print("-" * 40)
print(f"Total Test Images: {total_tests}")
print(f"Correctly Classified: {correct_predictions}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Testing Time: {end_time - start_time:.2f} seconds")
print("-" * 40)

# NOTE: To deploy this on the MCU, print or export the 'digit_templates' dictionary
# and paste it into your microcontroller code (receive_and_detect.py).

Splitting data: 50,000 Train and 20,000 Test...
Resizing images to 16x16 and Binarizing...
Training (Extracting 12 templates per digit)...

--- Testing on unseen images ---
Processed 5000 / 20000 images...
Processed 10000 / 20000 images...
Processed 15000 / 20000 images...
Processed 20000 / 20000 images...
----------------------------------------
Total Test Images: 20000
Correctly Classified: 17411
Accuracy: 87.06%
Testing Time: 4.80 seconds
----------------------------------------


In [ ]:
# ==========================================
# 6. Export Templates for Microcontroller
# ==========================================
# Export the trained templates as a formatted Python dictionary string.
# You can copy the console output and paste it directly into 'receive_and_detect.py' on your MCU.

print("DIGIT_TEMPLATES = {")
for digit, templates in digit_templates.items():
    print(f"    {digit}: [")
    for template in templates:
        # Convert the four 64-bit integers into hexadecimal format (0x...) 
        # for better readability and standard embedded memory representation.
        parts_hex = [f"0x{p:016x}" for p in template]
        print(f"        [{', '.join(parts_hex)}],")
    print("    ],")
print("}")

DIGIT_TEMPLATES = {
    0: [
        [0x00000000000007c0, 0x0ff00c181808180c, 0x180c180c180c0c18, 0x07f001c000000000],
        [0x00000000000000f8, 0x01f8038c070c0e0c, 0x0c081818183018e0, 0x1f800e0000000000],
        [0x00000000000001e0, 0x03f0031006180418, 0x0c180c180c300c60, 0x0fc0078000000000],
        [0x0000000000000030, 0x00f801fc038c0618, 0x0c18183018e01fc0, 0x1f00000000000000],
        [0x00000000000001f0, 0x03f807180c0c0c0c, 0x180c180818181830, 0x0fe0078000000000],
        [0x0000000000000000, 0x03f007f80c0c180c, 0x180c180c18381ff0, 0x0780000000000000],
        [0x00000000000003c0, 0x076006300c100c18, 0x0c180c180c180630, 0x07e001c000000000],
        [0x00000000000001c0, 0x03e0032002300630, 0x0430043004200660, 0x07c0038000000000],
        [0x00000000000003e0, 0x07f00ff80e381c18, 0x1c1c1c181c380ff0, 0x07f003c000000000],
        [0x0000000000000070, 0x00f001f003300330, 0x063004600c600dc0, 0x0f80060000000000],
        [0x00000000006001f0, 0x03f807980e180c18, 0x1c18183818701fe0, 0x